# 07 Next month's dengue: forecasting with TimesFM — notebook

Doc Mon needs numbers for the Local Health Board: how many dengue cases should Santa Luz plan for in the next eight weeks? In this notebook you will:

1. look at eight years of weekly dengue counts (`data/synthetic/ph/santa_luz_dengue_weekly.csv`, **synthetic**);
2. build two simple **baselines** first — they are the bar any model must clear;
3. run **TimesFM 3.0**, a zero-shot time-series foundation model, in a fair **rolling-origin backtest**;
4. watch what it does when the outbreak takes off, and whether rainfall helps;
5. make the real forecast from 2026-W39 — with an uncertainty band — and decide how far to trust it.

> **Two paths, same numbers.** On Google Colab (or any computer where `timesfm` is installed) the cells below really call TimesFM 3.0. When this website is built, the 1.3 GB model is not downloaded: `epi_learning.tsfm` replays forecasts saved from a real TimesFM 3.0 run of this same notebook (`data/tsfm_cache/ph07_dengue.json`). Every forecasting cell prints which path it took.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    sys.path.insert(0, '/content/python4epi/src')
    !pip install -q -e .

In [ ]:
# Extra Colab step: install TimesFM 3.0 (about 2 minutes; the weights download later, ~1.3 GB)
# On your own computer with uv:  uv sync --group timesfm
import sys
if 'google.colab' in sys.modules:
    !pip install -q "timesfm[torch]>=3.0.2"

## Step 1 — The weekly dengue series

One row per **morbidity week** (ISO week, Monday to Sunday), from 2019-W01 to 2026-W39, with the number of dengue cases by week of onset and the rainfall at the city weather station. We keep the cases as a plain NumPy array `y`: forecasting models want a list of numbers in time order.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from epi_learning import tsfm

weekly = pd.read_csv("data/synthetic/ph/santa_luz_dengue_weekly.csv", parse_dates=["week_start"])
y = weekly["cases"].to_numpy(dtype=float)          # the target series
rain = weekly["rainfall_mm"].to_numpy(dtype=float)  # a possible helper series (covariate)
week_label = weekly["morbidity_week"].to_numpy()
row_of = {label: i for i, label in enumerate(week_label)}  # "2026-W31" -> row number

print(f"{len(weekly)} weeks: {week_label[0]} to {week_label[-1]}")
print(weekly.tail(6)[["morbidity_week", "cases", "rainfall_mm"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(weekly["week_start"], y, color="#0038A8", linewidth=1.4)
ax.axvspan(pd.Timestamp("2026-07-13"), weekly["week_start"].iloc[-1], color="#CE1126", alpha=0.12,
           label="After Typhoon Amihan")
ax.set_ylabel("Dengue cases per week")
ax.set_title("Santa Luz City (synthetic): weekly dengue cases, 2019-W01 to 2026-W39")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## Step 2 — Baselines first

Before any clever model, Maya asks Tala for the two simplest forecasts a tired DSO could make on paper:

> **Ask Tala:** *I have a NumPy array `y` of weekly dengue counts in time order (made-up example: `[5, 9, 11, 4, ...]`, more than 52 weeks long). For a forecast made at row `o` (we know `y[:o]` only) I want the next `H = 8` weeks from two baselines: (1) "last value" — repeat `y[o-1]` eight times; (2) "seasonal naive" — copy the same eight weeks of last year, `y[o-52 : o-52+8]`. Do it for a list of origins and give me the mean absolute error (MAE) against the true values. Add a check I can do by hand.*

Tala's answer is the cell below. The **origins** are every second week from 2023-W01 to 2026-W31: 94 forecasts, each 8 weeks long, each using only the past. This is a **rolling-origin backtest**: we pretend to stand at each origin, forecast, and only then look at what really happened.

In [ ]:
H = 8                                             # forecast horizon: 8 weeks
first, last = row_of["2023-W01"], row_of["2026-W39"] + 1
origins = list(range(first, last - H + 1, 2))     # every 2nd week; each forecast ends by 2026-W39
actual = np.stack([y[o:o + H] for o in origins])  # shape (94, 8): what really happened

last_value = np.stack([np.repeat(y[o - 1], H) for o in origins])
seasonal_naive = np.stack([y[o - 52:o - 52 + H] for o in origins])  # no ISO week 53 after 2020


def mae(pred, truth=actual):
    # Mean absolute error: the average size of the miss, in cases per week
    return float(np.mean(np.abs(np.asarray(pred) - np.asarray(truth))))


print(f"{len(origins)} origins, first forecast starts {week_label[origins[0]]}, "
      f"last starts {week_label[origins[-1]]}")
print(f"MAE last value:     {mae(last_value):.2f} cases/week")
print(f"MAE seasonal naive: {mae(seasonal_naive):.2f} cases/week")

**Check it.** Never trust a backtest you have not poked. We take one origin and check by hand that (a) the "last value" forecast really repeats the week before the origin, (b) "seasonal naive" really copies the same weeks one year earlier, and (c) no forecast uses a week at or after its own origin.

In [ ]:
k = 10                                   # any origin will do
o = origins[k]
print("Forecast made after week:", week_label[o - 1], "| first forecast week:", week_label[o])
print("Last value repeats", y[o - 1], "->", last_value[k][:3], "...")
print("Seasonal naive copies", week_label[o - 52], "...", week_label[o - 52 + H - 1], "->", seasonal_naive[k])

assert np.all(last_value[k] == y[o - 1])
assert week_label[o - 52][5:] == week_label[o][5:], "same ISO week one year earlier"
assert np.array_equal(actual[k], y[o:o + H])
assert all(len(c) == o for c, o in zip(tsfm.rolling_origin_contexts(y, origins), origins)), "history must stop at the origin"
print("All checks passed.")

## Step 3 — Load TimesFM 3.0

TimesFM is a **time-series foundation model**: Google trained it once on a very large collection of time series, and we use it **zero-shot** — no training on our data, just "here is the history, give me the next 8 weeks". It returns a median forecast and nine **quantiles** (0.1 … 0.9); the 0.1 and 0.9 quantiles bound an **80% prediction interval**.

The official API is three lines (`ModelConfig`, `TimesFM3Evaluator`, `predict_batch`). If `timesfm` is not installed, this cell switches to the cache instead and says so.

In [ ]:
CACHE = tsfm.cache_path("ph07_dengue")   # data/tsfm_cache/ph07_dengue.json
MODE = tsfm.resolve_mode()               # "model" if timesfm is installed, else "cache"

if MODE == "cache":
    model = None
    print("Path = CACHE: TimesFM is not run in this session; forecasts are replayed from a real run.")
    print(tsfm.describe_cache(CACHE))
else:
    from timesfm3 import ModelConfig, TimesFM3Evaluator

    config = ModelConfig(
        checkpoint_path="google/timesfm-3.0-pytorch",  # weights on Hugging Face (~1.3 GB, first run only)
        per_core_batch_size=32,                        # series per forward pass
        device=tsfm.default_device(),                  # "cuda" on a GPU runtime, else "cpu"
    )
    model = TimesFM3Evaluator(config)
    print("Path = MODEL: TimesFM 3.0 loaded on", tsfm.default_device())

## Step 4 — The same backtest with TimesFM

Exactly the same 94 origins, the same 8-week horizon, the same MAE. `tsfm.rolling_origin_contexts(y, origins)` returns `y[:o]` for each origin — the history the model is allowed to see. With a model, `tsfm.forecast_batch` simply calls `model.predict_batch(contexts, horizon=8, return_quantiles=True)`; on CPU this takes a few minutes.

We score two periods separately: the **ordinary years 2023–2025** and **2026**, the year of the typhoon. A single average would hide the difference.

In [ ]:
contexts = tsfm.rolling_origin_contexts(y, origins)
bt = tsfm.forecast_batch(model, contexts, H, cache_key="backtest_cases", cache_file=CACHE)
tfm = bt.forecast                                   # median forecasts, shape (94, 8)
lo, hi = bt.quantile(0.1), bt.quantile(0.9)         # 80% prediction interval

year_of_end = weekly["iso_year"].to_numpy()[[o + H - 1 for o in origins]]
periods = {"2023-2025": year_of_end <= 2025, "2026": year_of_end == 2026}

rows = []
for period, sel in periods.items():
    for name, pred in [("Last value", last_value), ("Seasonal naive", seasonal_naive), ("TimesFM 3.0", tfm)]:
        err = np.abs(pred[sel] - actual[sel])
        rows.append({"period": period, "method": name, "origins": int(sel.sum()),
                     "MAE": err.mean(), "MAE week 1": err[:, 0].mean(), "MAE week 8": err[:, -1].mean()})
    rows.append({"period": period, "method": "TimesFM 80% interval coverage",
                 "origins": int(sel.sum()), "MAE": tsfm.coverage(actual[sel], lo[sel], hi[sel])})
score = pd.DataFrame(rows)
print(score.round(2).to_string(index=False))

### Reading the scoreboard

- **Ordinary years (2023–2025, 75 forecasts):** TimesFM's MAE is **3.50** cases per week, against **4.31** for seasonal naive and **4.76** for last value. Its 80% interval contained the truth **79%** of the time — about what an honest 80% band should do. With no training at all, that is a useful forecaster.
- **2026 (19 forecasts ending in 2026):** TimesFM's MAE is **5.32**, seasonal naive's **5.21**. Here the model is **no better than copying last year**. It is the best method for next week (MAE 1.97 in week 1) and falls behind by week 8 (10.08 vs 8.68 for seasonal naive). The band covered only **72%**.
- Lesson: a single overall score would have hidden this. Always score the unusual period on its own — that is the period the Local Health Board cares about.

## Tala got it wrong: the forecast that had already seen the answer

Maya asked Tala for "a better baseline: the average of the same week in all years". Tala's code scored beautifully — an MAE far below everything above. Doc Mon looked for ten seconds and asked: *"Which years went into that average?"* All of them — **including the very weeks being forecast**. That is **leakage**: the forecast was shown the answer. The cell below reproduces Tala's version, then the honest one that only averages years **before** the origin.

A second, quieter way to fool yourself is to report only the **best** week or the best origin. We print that too, next to the honest average.

In [ ]:
iso_week = weekly["iso_week"].to_numpy()
iso_year = weekly["iso_year"].to_numpy()

# Tala's version (WRONG): average of the same ISO week over ALL years, test years included
all_years_mean = weekly.groupby("iso_week")["cases"].mean()
leaky = np.stack([all_years_mean.loc[iso_week[o:o + H]].to_numpy() for o in origins])

# Honest version: only the three full years before the origin's year
honest = []
for o in origins:
    past = weekly[(iso_year >= iso_year[o] - 3) & (iso_year < iso_year[o])]
    honest.append(past.groupby("iso_week")["cases"].mean().reindex(iso_week[o:o + H]).to_numpy())
honest = np.stack(honest)

print(f"Same-week average, all years (LEAKY):   MAE {mae(leaky):.2f}  <- looks great, means nothing")
print(f"Same-week average, 3 past years only:   MAE {mae(honest):.2f}")
print(f"TimesFM 3.0 (only ever saw the past):   MAE {mae(tfm):.2f}")

per_origin = np.abs(tfm - actual).mean(axis=1)
print(f"\nCherry-picking: TimesFM's best origin has MAE {per_origin.min():.2f} "
      f"({week_label[origins[int(per_origin.argmin())]]}), its worst {per_origin.max():.2f} "
      f"({week_label[origins[int(per_origin.argmax())]]}); the honest average is {per_origin.mean():.2f}.")

## Step 5 — The hard moment: the outbreak takes off

The weeks that matter most are the ones where the curve turns. We give TimesFM the history up to 2026-W30, W31, W32 and W34, ask for the next 8 weeks each time, and draw the forecasts on top of what really happened. (2026-W32 is the week dengue first crossed the epidemic threshold in [chapter 06](../06_signals_and_thresholds.md).)

In [ ]:
takeoff_weeks = ["2026-W30", "2026-W31", "2026-W32", "2026-W34"]
takeoff_origins = [row_of[w] + 1 for w in takeoff_weeks]       # history includes that week
take = tsfm.forecast_batch(model, tsfm.rolling_origin_contexts(y, takeoff_origins), H,
                           cache_key="takeoff", cache_file=CACHE)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.6), sharey=True)
show = slice(row_of["2026-W20"], len(y))
for ax, k, w in zip(axes, range(4), takeoff_weeks):
    o = takeoff_origins[k]
    steps = np.arange(o, o + H)
    ax.bar(np.arange(len(y))[show], y[show], color="#9bbaff", label="Reported cases")
    ax.plot(steps, take.forecast[k], color="#CE1126", marker="o", ms=3, label="TimesFM median")
    ax.fill_between(steps, take.quantile(0.1)[k], take.quantile(0.9)[k], color="#CE1126", alpha=0.18,
                    label="80% interval")
    ax.axvline(o - 0.5, color="#1c1b29", linestyle="--", linewidth=1)
    ax.set_title(f"Knows data up to {w}", fontsize=10)
    ax.set_xticks([row_of["2026-W24"], row_of["2026-W32"], row_of["2026-W39"]], ["W24", "W32", "W39"])
    known = min(H, len(y) - o)
    inside = tsfm.coverage(y[o:o + known], take.quantile(0.1)[k][:known], take.quantile(0.9)[k][:known])
    print(f"Up to {w}: median next 4 weeks {np.round(take.forecast[k][:4]).astype(int)}, "
          f"actual {y[o:o + 4].astype(int)}, actual inside the 80% band: {inside:.0%} of {known} weeks")
axes[0].set_ylabel("Dengue cases per week")
axes[0].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## Step 6 — Would rainfall help?

In this synthetic series rainfall **leads** cases: the correlation is highest when rainfall is shifted by 7 weeks (see the data README). TimesFM 3.0 accepts **covariates**. `past_only_covariates` gives it rainfall up to the origin — the rain of Typhoon Amihan (2026-W29) is already in the history when we forecast from W31. Does the model use it? Same backtest, same take-off origins, now with rainfall.

In [ ]:
rain_past = [rain[None, :o] for o in origins]            # one (1, T) array per origin
bt_rain = tsfm.forecast_batch(model, contexts, H, cache_key="backtest_rain", cache_file=CACHE,
                              past_only_covariates=rain_past)
take_rain = tsfm.forecast_batch(model, tsfm.rolling_origin_contexts(y, takeoff_origins), H,
                                cache_key="takeoff_rain", cache_file=CACHE,
                                past_only_covariates=[rain[None, :o] for o in takeoff_origins])

for period, sel in periods.items():
    print(f"{period}: MAE cases only {mae(tfm[sel], actual[sel]):.2f} | "
          f"with rainfall {mae(bt_rain.forecast[sel], actual[sel]):.2f}")
k = takeoff_weeks.index("2026-W31")
print("\nFrom 2026-W31, next 8 weeks")
print("  actual          ", y[takeoff_origins[k]:takeoff_origins[k] + H].astype(int))
print("  cases only      ", np.round(take.forecast[k]).astype(int))
print("  with rainfall   ", np.round(take_rain.forecast[k]).astype(int))

## Step 7 — The forecast Doc Mon asked for: 2026-W40 to W47

Now the real thing: all the history up to 2026-W39, the next 8 weeks, with the 80% interval. But remember [chapter 06](../06_signals_and_thresholds.md): **W39 is incomplete** — at the 30 September freeze, cases with onset that week were still being reported. So we also run the model on the history **without W39** and compare. If the two disagree a lot, the latest weeks are steering the forecast and you should say so.

In [ ]:
if model is not None:
    # The genuine TimesFM 3.0 call, once, without the helper:
    out = next(model.predict_batch([y.astype(np.float32)], horizon=H, return_quantiles=True))
    print("predict_batch -> forecast", out.forecast.shape, "quantiles", out.quantiles.shape)

now = tsfm.forecast_batch(model, [y, y[:-1]], H, cache_key="forecast_now", cache_file=CACHE)
future_weeks = [f"2026-W{w}" for w in range(40, 48)]
table = pd.DataFrame({
    "week": future_weeks,
    "low (q0.1)": now.quantile(0.1)[0],
    "median": now.forecast[0],
    "high (q0.9)": now.quantile(0.9)[0],
    "median if W39 dropped": now.forecast[1][1:].tolist() + [np.nan],
}).round(0)
print(table.to_string(index=False))
print(f"\nFrom data up to W38 the model expected {now.forecast[1][0]:.0f} cases in W39; "
      f"{y[-1]:.0f} were reported by the freeze.")
print(f"Sum of the weekly medians, W40-W47: {now.forecast[0].sum():.0f} cases")
print(f"Same 8 weeks last year (2025-W40 to W47): {y[len(y) - 52:len(y) - 52 + H].sum():.0f} cases")

fig, ax = plt.subplots(figsize=(10, 3.8))
past = np.arange(row_of["2026-W20"], len(y))
ax.bar(past, y[past], color="#9bbaff", label="Reported cases (W39 incomplete)")
steps = np.arange(len(y), len(y) + H)
ax.plot(steps, now.forecast[0], color="#CE1126", marker="o", ms=3, label="TimesFM median")
ax.fill_between(steps, now.quantile(0.1)[0], now.quantile(0.9)[0], color="#CE1126", alpha=0.18,
                label="80% prediction interval")
ax.plot(np.arange(len(y) - 1, len(y) - 1 + H), now.forecast[1], color="#2f7d32", linestyle="--",
        label="Median if W39 is left out")
ax.set_xticks([row_of["2026-W24"], row_of["2026-W32"], row_of["2026-W39"], len(y) + 7],
              ["W24", "W32", "W39", "W47"])
ax.set_ylabel("Dengue cases per week")
ax.set_title("Forecast from 2026-W39 (synthetic data)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## Your turn

These use the arrays you already have — no new model calls, so they work on the website and on Colab.

1. **Which horizon is hardest?** Print the MAE of each method for each horizon week 1…8 (hint: `np.abs(tfm - actual).mean(axis=0)` gives eight numbers). Where does TimesFM lose its lead over seasonal naive in 2026?
2. **Is the 80% band honest?** For 2023–2025, count the weeks where the truth fell **above** the band and **below** it (hint: `(actual > hi)[sel].mean()`). An honest band misses about 10% on each side.
3. **Plan, not prophecy.** Using `table`, write one sentence for the Local Health Board that gives a range, not a single number, and names one thing that would make you revise it.

In [ ]:
# 1. MAE by horizon week (starter: TimesFM, 2026 origins)
sel = periods["2026"]
print("TimesFM, 2026, MAE by week ahead:", np.abs(tfm[sel] - actual[sel]).mean(axis=0).round(1))
# ... now do the same for seasonal_naive and compare

# 2. Band misses (starter)
sel = periods["2023-2025"]
print("Share of truth above the band, 2023-2025:", round(float((actual > hi)[sel].mean()), 2))
# ... and below?

## Before you use this at work: the licence

- TimesFM's **code** is Apache-2.0, but the **TimesFM 3.0 weights** (`google/timesfm-3.0-pytorch`) are under the **TimesFM Non-Commercial License v1.0**: research, evaluation and teaching, **no commercial and no production use**. Read it yourself: [TimesFM 3.0 licence on Hugging Face](https://huggingface.co/google/timesfm-3.0-pytorch/blob/main/LICENSE).
- The **TimesFM 2.5** weights are Apache-2.0. Google offers commercial and production use of TimesFM through Google Cloud.
- What that means for a government health office: learning, backtesting old data and exploring — like this notebook — fits. Using 3.0 forecasts in a routine early-warning system, a public bulletin or routine operational decisions does not. **Check the licence (and your office's rules) before any operational use.** This is not legal advice.
- The cached forecasts in `data/tsfm_cache/ph07_dengue.json` are model outputs produced under that licence for this non-commercial teaching material.

**What you learned:** beat the baselines in a fair backtest before you believe a model; score the hard weeks separately; never let the forecast see the answer; and a model cannot see a typhoon, a new serotype, or the cases that have not been reported yet.